[Reference](https://medium.com/data-science-collective/hybrid-search-for-rag-bm25-vectors-when-each-wins-402f24abaeea)

In [1]:
import bm25s
import Stemmer

def build_bm25_retriever(corpus: list[str]) -> bm25s.BM25:
    # We use a stemmer to match variations like "running" and "run"
    stemmer = Stemmer.Stemmer("english")

    # Tokenize the corpus and remove common stop words
    tokens = bm25s.tokenize(corpus, stopwords="en", stemmer=stemmer)

    # Initialize and index the BM25 model
    retriever = bm25s.BM25(corpus=corpus)
    retriever.index(tokens)

    return retriever

def bm25_search(retriever: bm25s.BM25, query: str, k: int = 5) -> list[dict]:
    stemmer = Stemmer.Stemmer("english")
    q_tokens = bm25s.tokenize(query, stemmer=stemmer)

    # Retrieve the top-k documents and their scores
    docs, scores = retriever.retrieve(q_tokens, k=k)

    results = []
    for i in range(docs.shape[1]):
        results.append({
            "content": docs[0, i],
            "score": float(scores[0, i])
        })
    return results

In [2]:
from qdrant_client import QdrantClient, models
from openai import OpenAI

def build_vector_index(chunks: list[dict], collection_name: str = "docs") -> QdrantClient:
    # Using in-memory mode for demonstration
    client = QdrantClient(":memory:")

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=1536,
            distance=models.Distance.COSINE
        ),
    )

    oai = OpenAI()
    texts = [c["content"] for c in chunks]

    # Generate embeddings for all chunks
    resp = oai.embeddings.create(input=texts, model="text-embedding-3-small")
    vectors = [e.embedding for e in resp.data]

    # Insert into the database with payload metadata
    points = []
    for i, chunk in enumerate(chunks):
        points.append(
            models.PointStruct(
                id=i,
                vector=vectors[i],
                payload={"content": chunk["content"], **chunk.get("meta", {})}
            )
        )

    client.upsert(collection_name=collection_name, points=points)
    return client

def vector_search(client: QdrantClient, query: str, collection_name: str = "docs", k: int = 5) -> list[dict]:
    oai = OpenAI()
    q_vec = oai.embeddings.create(input=[query], model="text-embedding-3-small").data[0].embedding

    hits = client.query_points(
        collection_name=collection_name,
        query=q_vec,
        limit=k,
    ).points

    results = []
    for hit in hits:
        results.append({
            "content": hit.payload["content"],
            "score": hit.score,
            "id": hit.id
        })
    return results

# How Hybrid Search Actually Works

In [3]:
def reciprocal_rank_fusion(*result_lists: list[dict], k: int = 60, top_n: int = 5) -> list[dict]:
    scores: dict[str, float] = {}
    best_docs: dict[str, dict] = {}

    for results in result_lists:
        for rank, result in enumerate(results):
            # We need a unique identifier to deduplicate chunks across lists
            # In a real system, use the chunk ID. Here we use the first 100 chars.
            doc_id = result.get("id", result["content"][:100])

            if doc_id not in scores:
                scores[doc_id] = 0.0

            # Add the RRF penalty based on the rank
            scores[doc_id] += 1.0 / (k + rank + 1)

            # Keep the document payload for the final output
            if doc_id not in best_docs:
                best_docs[doc_id] = result

    # Sort the documents by their new RRF score in descending order
    ranked_ids = sorted(scores, key=scores.__getitem__, reverse=True)[:top_n]

    final_results = []
    for doc_id in ranked_ids:
        doc = best_docs[doc_id].copy()
        doc["rrf_score"] = scores[doc_id]
        final_results.append(doc)

    return final_results

In [4]:
from dataclasses import dataclass

@dataclass
class EvalCase:
    query: str
    expected_substring: str
    query_type: str  # "identifier", "conceptual", or "mixed"

def evaluate_retrievers(cases: list[EvalCase], retrievers: dict[str, callable], k: int = 5) -> dict:
    report = {name: {"total_hits": 0, "by_type": {}} for name in retrievers}

    for case in cases:
        for name, search_fn in retrievers.items():
            # Execute the search function
            results = search_fn(case.query, k=k)

            # Check if the expected answer is in the top-k chunks
            top_contents = [r["content"] for r in results]
            found = any(case.expected_substring in content for content in top_contents)

            # Record the metrics
            report[name]["total_hits"] += int(found)

            q_type = case.query_type
            if q_type not in report[name]["by_type"]:
                report[name]["by_type"][q_type] = {"hits": 0, "total": 0}

            report[name]["by_type"][q_type]["total"] += 1
            report[name]["by_type"][q_type]["hits"] += int(found)

    # Calculate final hit rates
    total_cases = len(cases)
    for name in report:
        report[name]["overall_hit_rate"] = report[name]["total_hits"] / total_cases if total_cases else 0
        for q_type, stats in report[name]["by_type"].items():
            stats["hit_rate"] = stats["hits"] / stats["total"] if stats["total"] else 0

    return report